In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# =========================
# LOAD DATA
# =========================
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

X = train.drop(['id', 'Irrigation_Need'], axis=1)
y = train['Irrigation_Need']
X_test = test.drop(['id'], axis=1)

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# =========================
# ENCODING (IMPORTANT)
# =========================
X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

# Reduce memory
X = X.astype(np.float32)
X_test = X_test.astype(np.float32)

# =========================
# LIGHTGBM (CV)
# =========================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

lgb_test_preds = np.zeros((X_test.shape[0], len(le.classes_)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_encoded)):
    print(f"\nFold {fold+1}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    model = LGBMClassifier(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    lgb_test_preds += model.predict_proba(X_test) / skf.n_splits

# =========================
# CATBOOST (NO CV → SAVES MEMORY)
# =========================
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    verbose=0,
    random_state=42
)

cat_model.fit(X, y_encoded)
cat_test_preds = cat_model.predict_proba(X_test)

# =========================
# ENSEMBLE (WEIGHTED)
# =========================
final_proba = (0.7 * lgb_test_preds) + (0.3 * cat_test_preds)

final_preds = np.argmax(final_proba, axis=1)
final_preds = le.inverse_transform(final_preds)

# =========================
# SAVE
# =========================
submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_preds
})

submission.to_csv('submission.csv', index=False)

print("Submission created!")


Fold 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043884 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2730
[LightGBM] [Info] Number of data points in the train set: 420000, number of used features: 43
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612

Fold 2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012566 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2729
[LightGBM] [Info] Number of data points in the train set: 420000, number of used features: 43
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612

Fold 3
[LightGB